In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project='')

In [ ]:
import geemap.core as geemap

In [ ]:
import geemap
import ee

ee.Initialize()

california = ee.FeatureCollection("TIGER/2018/States").filter(ee.Filter.eq('NAME', 'California'))

dataset = ee.ImageCollection("LANDSAT/LC09/C02/T1").filterDate('2023-08-15', '2023-11-17').filterBounds(california) # Smith River Complex fire dates

def apply_scale_factors(image):
    optical_bands = image.select('B6').multiply(0.0000275).add(-0.2)
    thermal_bands_b10 = image.select('B10').multiply(0.00341802).add(149.0)
    thermal_bands_b11 = image.select('B11').multiply(0.00341802).add(149.0)
    return image.addBands(optical_bands, None, True).addBands(thermal_bands_b10, None, True).addBands(thermal_bands_b11, None, True)

dataset = dataset.map(apply_scale_factors)


median_image = dataset.median()
band_names = median_image.bandNames()
print(f"Band names: {band_names.getInfo()}")


# Visualization
visualization = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 30000,
}

m = geemap.Map()
m.set_center(-119.4179, 36.7783, 6)  # Center map on California
m.addLayer(median_image, visualization, 'True Color (432)')
m.addLayer(california.style(color='red', fillColor='00000000'), {}, 'California Boundary')
m


Band names: ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B9', 'B10', 'B11', 'QA_PIXEL', 'QA_RADSAT', 'SAA', 'SZA', 'VAA', 'VZA']


Map(center=[36.7783, -119.4179], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=Search…

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive/', force_remount=True)
os.chdir('/content/drive/My Drive/')
print(os.getcwd())


In [ ]:
import ee
import geemap


ee.Initialize()

# Define California
california = ee.FeatureCollection("TIGER/2018/States").filter(ee.Filter.eq('NAME', 'California'))

dataset = ee.ImageCollection("LANDSAT/LC09/C02/T1").filterDate('2023-08-15', '2023-11-17').filterBounds(california) # Smith River Complex fire dates


def apply_scale_factors(image):
    optical_bands = ee.Algorithms.If(
        image.bandNames().contains('B6'),
        image.select('B6'),
        ee.Image(0)
    )
    thermal_bands_b10 = ee.Algorithms.If(
        image.bandNames().contains('B10'),
        image.select('B10'),
        ee.Image(0)
    )
    thermal_bands_b11 = ee.Algorithms.If(
        image.bandNames().contains('B11'),
        image.select('B11'),
        ee.Image(0)
    )
    return ee.Image(optical_bands).addBands(ee.Image(thermal_bands_b10)).addBands(ee.Image(thermal_bands_b11))

dataset = dataset.map(apply_scale_factors)
grid = ee.FeatureCollection(california.geometry().coveringGrid('EPSG:4326', scale=25000))
grid_list = grid.toList(grid.size())
grid_size = grid.size().getInfo()
print(f"Number of tiles in the grid: {grid_size}")

for i in range(grid_size):
    region = ee.Feature(grid_list.get(i)).geometry()
    clipped_image = dataset.median().clip(region).select(['B6', 'B10', 'B11'])  # Selecting only the desired bands

    task = ee.batch.Export.image.toDrive(
        image=clipped_image,
        description=f'SmithRiverComplex_Tile_{i}',
        folder='SmithRiverComplexFire3',
        region=region,
        scale=100,  # Metres per pixel
        maxPixels=1e10
    )
    print(f"Exporting tile {i}...")
    task.start()


Number of tiles in the grid: 980
Exporting tile 0...
Exporting tile 1...
Exporting tile 2...
Exporting tile 3...
Exporting tile 4...
Exporting tile 5...
Exporting tile 6...
Exporting tile 7...
Exporting tile 8...
Exporting tile 9...
Exporting tile 10...
Exporting tile 11...
Exporting tile 12...
Exporting tile 13...
Exporting tile 14...
Exporting tile 15...
Exporting tile 16...
Exporting tile 17...
Exporting tile 18...
Exporting tile 19...
Exporting tile 20...
Exporting tile 21...
Exporting tile 22...
Exporting tile 23...
Exporting tile 24...
Exporting tile 25...
Exporting tile 26...
Exporting tile 27...
Exporting tile 28...
Exporting tile 29...
Exporting tile 30...
Exporting tile 31...
Exporting tile 32...
Exporting tile 33...
Exporting tile 34...
Exporting tile 35...
Exporting tile 36...
Exporting tile 37...
Exporting tile 38...
Exporting tile 39...
Exporting tile 40...
Exporting tile 41...
Exporting tile 42...
Exporting tile 43...
Exporting tile 44...
Exporting tile 45...
Exporting t

In [ ]:
california = ee.FeatureCollection("TIGER/2018/States").filter(ee.Filter.eq('NAME', 'California'))

dataset = ee.ImageCollection("LANDSAT/LC09/C02/T1").filterDate('2023-08-15', '2023-11-17').filterBounds(california) # Smith River Complex fire dates

def apply_scale_factors(image):
    optical_bands = image.select('B6').multiply(0.0000275).add(-0.2)
    thermal_bands_b10 = image.select('B10').multiply(0.00341802).add(149.0)
    thermal_bands_b11 = image.select('B11').multiply(0.00341802).add(149.0)
    return image.addBands(optical_bands, None, True).addBands(thermal_bands_b10, None, True).addBands(thermal_bands_b11, None, True)

dataset = dataset.map(apply_scale_factors)


median_image = dataset.median()


# Visualization
visualization = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 30000,
}

m = geemap.Map()
m.set_center(-119.4179, 36.7783, 6)  # Center map on California
m.addLayer(median_image, visualization, 'True Color (432)')
m.addLayer(california.style(color='red', fillColor='00000000'), {}, 'California Boundary')
m.addLayer(grid.style(color='blue', fillColor='00000000', width=2), {}, 'Grid Tiles')
m


Band names: ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B9', 'B10', 'B11', 'QA_PIXEL', 'QA_RADSAT', 'SAA', 'SZA', 'VAA', 'VZA']


Map(center=[36.7783, -119.4179], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=Search…

In [ ]:
pip install rasterio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 47.3 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

Mounted at /content/drive/


In [ ]:
import rasterio
import matplotlib.pyplot as plt
import os
from google.colab import drive
import numpy as np

os.chdir('/content/drive/My Drive/')
print(os.getcwd())

folder_path = "SmithRiverComplexFire"
file_paths = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.tif')]

for i, file_path in enumerate(file_paths):
    with rasterio.open(file_path) as src:
        band_6 = src.read(1)
        max_band_6 = np.nanmax(band_6)
        print(f"Tile {i} - Max Pixel Values:")
        print(f"Band 6: {max_band_6}")



